In [1]:
import sys
import subprocess

In [2]:
import html
import re
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import spacy

from tqdm.auto import tqdm
from rapidfuzz import process
from rapidfuzz.distance import DamerauLevenshtein

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

from IPython.display import display


SEED = 42

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

pd.set_option("display.max_colwidth", 100)

/home/mlguest/miniforge3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import html
import re
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import spacy

from tqdm.auto import tqdm
from rapidfuzz import process
from rapidfuzz.distance import DamerauLevenshtein

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

from IPython.display import display


SEED = 42

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

pd.set_option("display.max_colwidth", 100)

In [5]:
products = pd.read_csv("product.csv", sep="\t")
queries = pd.read_csv("query.csv", sep="\t")
labels = pd.read_csv("label.csv", sep="\t")

text_columns = [
    "product_name",
    "product_class",
    "category hierarchy",
    "product_description"
]

products[text_columns] = products[text_columns].fillna("")
queries["query"] = queries["query"].fillna("")

products = products.reset_index(drop=True)

assert products["product_id"].is_unique
assert queries["query_id"].is_unique

assert labels["product_id"].isin(products["product_id"]).all()
assert labels["query_id"].isin(queries["query_id"]).all()

# Удаляем повторения с одинаковой оценкой
labels = labels.drop_duplicates(
    ["query_id", "product_id", "label"]
).copy()

# После этого повторяющиеся пары имеют разные оценки
conflict_mask = labels.duplicated(
    ["query_id", "product_id"],
    keep=False
)

conflicting_labels = labels.loc[conflict_mask].copy()

conflict_count = conflicting_labels[
    ["query_id", "product_id"]
].drop_duplicates().shape[0]

print("Противоречивых пар:", conflict_count)

display(
    conflicting_labels.sort_values(["query_id", "product_id"])
)

# Исключаем все оценки спорных пар
labels = labels.loc[~conflict_mask].copy()

label_to_relevance = {
    "Irrelevant": 0,
    "Partial": 1,
    "Exact": 2
}

labels["relevance"] = labels["label"].map(label_to_relevance)

assert labels["relevance"].notna().all()
assert not labels.duplicated(["query_id", "product_id"]).any()

print("Товаров:", len(products))
print("Запросов:", len(queries))
print("Размеченных пар после очистки:", len(labels))

display(labels["label"].value_counts())
display(products.head(3))

Противоречивых пар: 14


,id,query_id,product_id,label
65,65,0,31306,Irrelevant
21650,21650,0,31306,Partial
1325,1325,12,34349,Exact
21761,21761,12,34349,Partial
2380,2380,24,696,Exact
21853,21853,24,696,Partial
11489,11489,89,24413,Exact
22346,22346,89,24413,Partial
13869,13869,106,1990,Partial
22448,22448,106,1990,Irrelevant


Товаров: 42994
Запросов: 480
Размеченных пар после очистки: 231859


label
Partial       145670
Irrelevant     60719
Exact          25470
Name: count, dtype: int64

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboards / Beds / Twin Beds,"good , deep sleep can be quite difficult to have in this busy age . fortunately , there ’ s an a...",overallwidth-sidetoside:64.7|dsprimaryproductstyle : modern|countryoforigin-additionaldetails : ...,15.0,4.5,15.0
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances / Pressure & Slow Cookers / Slow Cookers / Slow Sl...,"create delicious slow-cooked meals , from tender meat to flavorful veggies , with this easy-to-u...",capacityquarts:7|producttype : slow cooker|programmablecookingsettings : slow cook|overallheight...,100.0,2.0,98.0
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances / Pressure & Slow Cookers / Slow Cookers / Slow Sl...,"prepare home-cooked meals on any schedule with this essential slow cooker , featuring a dishwash...",features : keep warm setting|capacityquarts:6.5|programmablefoodsettings : soup|warrantylength:1...,208.0,3.0,181.0


In [6]:
dev_query_ids, test_query_ids = train_test_split(
    queries["query_id"].to_numpy(),
    test_size=0.2,
    random_state=SEED
)

print("Запросов для экспериментов:", len(dev_query_ids))
print("Тестовых запросов:", len(test_query_ids))

Запросов для экспериментов: 384
Тестовых запросов: 96


In [7]:
KEEP_WORDS = {
    "no", "not", "nor", "without",
    "above", "below", "under", "over"
}


def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("’", "'")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def doc_to_text(doc):
    tokens = []

    for token in doc:
        if token.is_space or token.is_punct:
            continue

        word = token.text.lower()

        if token.is_stop and word not in KEEP_WORDS:
            continue

        if not any(char.isalnum() for char in word):
            continue

        # Числа и артикулы с цифрами оставляем как есть
        if any(char.isdigit() for char in word):
            lemma = word
        else:
            lemma = token.lemma_.lower().strip() or word

        tokens.append(lemma)

    return " ".join(tokens)


def preprocess_many(texts, description):
    cleaned = (clean_text(text) for text in texts)

    docs = nlp.pipe(
        cleaned,
        batch_size=128,
        n_process=1
    )

    return [
        doc_to_text(doc)
        for doc in tqdm(
            docs,
            total=len(texts),
            desc=description
        )
    ]


@lru_cache(maxsize=4096)
def preprocess_query(text):
    return doc_to_text(nlp(clean_text(text)))

In [8]:
products["title_raw"] = (
    products["product_name"]
    + " "
    + products["product_class"]
    + " "
    + products["category hierarchy"]
)

products["title_processed"] = preprocess_many(
    products["title_raw"].tolist(),
    "Названия и категории"
)

products["description_processed"] = preprocess_many(
    products["product_description"].tolist(),
    "Описания"
)

display(
    products[
        ["product_name", "title_processed", "description_processed"]
    ].head(3)
)

Описания: 100%|██████████| 42994/42994 [01:55<00:00, 373.86it/s]


,product_name,title_processed,description_processed
0,solid wood platform bed,solid wood platform bed bed furniture bedroom furniture beds headboards beds twin bed,good deep sleep difficult busy age fortunately s antidote problem nice quality bed frame like ac...
1,all-clad 7 qt . slow cooker,clothe 7 qt slow cooker slow cookers kitchen tabletop small kitchen appliances pressure slow coo...,create delicious slow cook meal tender meat flavorful veggie easy use slow cooker unit feature n...
2,all-clad electrics 6.5 qt . slow cooker,clothe electric 6.5 qt slow cooker slow cookers kitchen tabletop small kitchen appliances pressu...,prepare home cook meal schedule essential slow cooker feature dishwasher safe insert 26 hour timer


In [9]:
def make_vectorizer():
    return TfidfVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=1,
        max_features=150_000,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32
    )


title_vectorizer = make_vectorizer()
description_vectorizer = make_vectorizer()

X_title = title_vectorizer.fit_transform(
    products["title_processed"]
)

X_description = description_vectorizer.fit_transform(
    products["description_processed"]
)

product_id_to_row = pd.Series(
    products.index,
    index=products["product_id"]
).to_dict()

print("Матрица названий:", X_title.shape)
print("Матрица описаний:", X_description.shape)

Матрица названий: (42994, 110366)
Матрица описаний: (42994, 150000)


In [10]:
word_pattern = re.compile(r"\b[a-zA-Z]+\b")

word_counts = Counter()

catalog_texts = (
    products["product_name"]
    + " "
    + products["product_class"]
    + " "
    + products["product_description"]
)

for text in catalog_texts:
    word_counts.update(
        word_pattern.findall(clean_text(text).lower())
    )

known_words = set(word_counts)

correction_candidates = sorted(
    word
    for word, count in word_counts.items()
    if count >= 5 and len(word) >= 4
)


@lru_cache(maxsize=10_000)
def correct_word(word):
    if len(word) < 5 or word in known_words:
        return word

    matches = process.extract(
        word,
        correction_candidates,
        scorer=DamerauLevenshtein.distance,
        score_cutoff=1,
        limit=2
    )

    if len(matches) == 1:
        return matches[0][0]

    return word


def correct_query(query):
    changes = []

    def replace_match(match):
        original = match.group(0)
        corrected = correct_word(original.lower())

        if corrected != original.lower():
            changes.append((original, corrected))
            return corrected

        return original

    # Не исправляем буквенные части артикулов с цифрами
    corrected = re.sub(
        r"\b[a-zA-Z]+\b",
        replace_match,
        clean_text(query)
    )

    return corrected, changes


print(correct_query("wooden cofffee table"))

('wooden coffee table', [('cofffee', 'coffee')])


In [11]:
def score_query(
    query,
    title_weight=0.7,
    fix_typos=False,
    candidate_rows=None
):
    if not 0 <= title_weight <= 1:
        raise ValueError("title_weight должен быть от 0 до 1")

    if fix_typos:
        used_query, changes = correct_query(query)
    else:
        used_query = clean_text(query)
        changes = []

    processed = preprocess_query(used_query)

    query_title = title_vectorizer.transform([processed])
    query_description = description_vectorizer.transform([processed])

    title_matrix = (
        X_title if candidate_rows is None
        else X_title[candidate_rows]
    )

    description_matrix = (
        X_description if candidate_rows is None
        else X_description[candidate_rows]
    )

    title_scores = (
        title_matrix @ query_title.T
    ).toarray().ravel()

    description_scores = (
        description_matrix @ query_description.T
    ).toarray().ravel()

    scores = (
        title_weight * title_scores
        + (1 - title_weight) * description_scores
    )

    return scores, used_query, processed, changes

In [12]:
def search(query, top_k=10, title_weight=0.7, fix_typos=True):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Введите непустой запрос")

    if top_k < 1:
        raise ValueError("top_k должен быть положительным")

    scores, used_query, processed, changes = score_query(
        query,
        title_weight=title_weight,
        fix_typos=fix_typos
    )

    # Товары без совпадений не включаем
    matching_rows = np.flatnonzero(scores > 0)

    order = matching_rows[
        np.argsort(-scores[matching_rows], kind="stable")
    ][:top_k]

    result = products.iloc[order][[
        "product_id",
        "product_name",
        "product_class",
        "product_description"
    ]].copy()

    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores[order]

    print("Исходный запрос:", query)
    print("Использованный запрос:", used_query)
    print("После лемматизации:", processed)

    if changes:
        print("Исправления:", changes)

    if len(result) == 0:
        print("Совпадений не найдено.")

    return result.reset_index(drop=True)

In [13]:
display(search("wooden coffee table", top_k=10))

Исходный запрос: wooden coffee table
Использованный запрос: wooden coffee table
После лемматизации: wooden coffee table


,rank,product_id,product_name,product_class,product_description,score
0,1,9364,diahann wooden coffee table,Coffee & Cocktail Tables,center your conversation space or entertainment ensemble around sleek style when you add this el...,0.475272
1,2,5434,wooden coffee table,Coffee & Cocktail Tables,a rustic coffee table featuring round wooden base in quatrefoil cutout design and round wooden t...,0.464696
2,3,5390,treadaway wooden coffee table,Coffee & Cocktail Tables,"not only does this coffee table offer years of practical function , but it also makes an impress...",0.435303
3,4,5371,forsman wooden coffee table,Coffee & Cocktail Tables,substantial storage and compact design of this wooden coffee table will give you a valid reason ...,0.426204
4,5,31465,wellkamp wooden coffee table,Coffee & Cocktail Tables,"an eye-catcher set of an elegant coffee table with modish wedge-shaped ottomans , this set is en...",0.393512
5,6,5356,bellin wooden coffee table,Coffee & Cocktail Tables,"sturdy , stylish and aesthetic this craftsmanship is perfect for modern decor . this solid woode...",0.372650
6,7,5359,aquinas wooden coffee table,Coffee & Cocktail Tables,"exhibited perfectly for your transitional interiors , this coffee/cocktail table will make a cla...",0.368510
7,8,5355,belhaven wooden coffee table,Coffee & Cocktail Tables,"keeping it traditional but attractive , this cocktail table will influence a warm appeal to your...",0.363724
8,9,5410,osuna wooden coffee table,Coffee & Cocktail Tables,upscale your living area or hallway bringing this product in a walnut brown finish . made unique...,0.361143
9,10,5338,jayla intricately carved wooden coffee table,Coffee & Cocktail Tables,"this intricately carved wooden coffee table features a scalloped top , wooden apron with antique...",0.356880


In [14]:
query_text_by_id = queries.set_index("query_id")["query"].to_dict()

label_groups = {
    query_id: group
    for query_id, group in labels.groupby("query_id", sort=False)
}


def evaluate_search(
    query_ids,
    title_weight=0.7,
    fix_typos=False,
    k=10
):
    results = []

    for query_id in tqdm(query_ids, desc="Оценка"):
        group = label_groups.get(query_id)

        if group is None or len(group) < 2:
            continue

        relevance = group["relevance"].to_numpy(dtype=float)

        # NDCG неинформативна без положительно размеченных товаров
        if relevance.max() == 0:
            continue

        candidate_rows = np.array([
            product_id_to_row[product_id]
            for product_id in group["product_id"]
        ])

        scores, _, _, _ = score_query(
            query_text_by_id[query_id],
            title_weight=title_weight,
            fix_typos=fix_typos,
            candidate_rows=candidate_rows
        )

        value = ndcg_score(
            relevance.reshape(1, -1),
            scores.reshape(1, -1),
            k=k,
            ignore_ties=False
        )

        results.append({
            "query_id": query_id,
            "query": query_text_by_id[query_id],
            f"NDCG@{k}": value
        })

    return pd.DataFrame(
        results,
        columns=["query_id", "query", f"NDCG@{k}"]
    )

In [15]:
dev_plain = evaluate_search(
    dev_query_ids,
    fix_typos=False
)

dev_corrected = evaluate_search(
    dev_query_ids,
    fix_typos=True
)

comparison = pd.DataFrame([
    {
        "Вариант": "TF-IDF без исправления",
        "NDCG@10": dev_plain["NDCG@10"].mean(),
        "Запросов": len(dev_plain)
    },
    {
        "Вариант": "TF-IDF с исправлением",
        "NDCG@10": dev_corrected["NDCG@10"].mean(),
        "Запросов": len(dev_corrected)
    }
])

display(comparison.round(4))

Оценка: 100%|██████████| 384/384 [00:00<00:00, 743.60it/s]


,Вариант,NDCG@10,Запросов
0,TF-IDF без исправления,0.7815,382
1,TF-IDF с исправлением,0.7817,382


In [16]:
display(
    dev_plain.sort_values("NDCG@10").head(10)
)

,query_id,query,NDCG@10
52,29,bathroom vanity knobs,0.000000
325,88,rose gold lounge,0.000000
276,14,beds that have leds,0.000000
268,135,low profile loveseat recliner,0.000000
135,304,merlyn 6,0.000000
183,476,hardwood beds,0.000000
219,246,clear shower curtain,0.000000
354,194,black frame 22 x 36,0.000000
174,409,teal chair,0.036682
117,384,dining table vinyl cloth,0.069431


BM25

In [18]:
from rank_bm25 import BM25Okapi

bm25_title = BM25Okapi(
    [text.split() for text in products["title_processed"]],
    k1=1.5,
    b=0.75
)

bm25_description = BM25Okapi(
    [text.split() for text in products["description_processed"]],
    k1=1.5,
    b=0.75
)

print("Индексы BM25 готовы")

Индексы BM25 готовы


In [19]:
def score_query_bm25(
    query,
    title_weight=0.7,
    fix_typos=False,
    candidate_rows=None
):
    if not 0 <= title_weight <= 1:
        raise ValueError("title_weight должен быть от 0 до 1")

    if fix_typos:
        used_query, changes = correct_query(query)
    else:
        used_query = clean_text(query)
        changes = []

    processed = preprocess_query(used_query)
    tokens = processed.split()

    if candidate_rows is None:
        title_scores = bm25_title.get_scores(tokens)
        description_scores = bm25_description.get_scores(tokens)
    else:
        rows = np.asarray(candidate_rows, dtype=int).tolist()

        title_scores = np.asarray(
            bm25_title.get_batch_scores(tokens, rows)
        )
        description_scores = np.asarray(
            bm25_description.get_batch_scores(tokens, rows)
        )

    scores = (
        title_weight * title_scores
        + (1 - title_weight) * description_scores
    )

    return scores, used_query, processed, changes

In [20]:
def search_bm25(
    query,
    top_k=10,
    title_weight=0.7,
    fix_typos=False
):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Введите непустой запрос")

    if top_k < 1:
        raise ValueError("top_k должен быть положительным")

    scores, used_query, processed, changes = score_query_bm25(
        query,
        title_weight=title_weight,
        fix_typos=fix_typos
    )

    matching_rows = np.flatnonzero(scores > 0)

    order = matching_rows[
        np.argsort(-scores[matching_rows], kind="stable")
    ][:top_k]

    result = products.iloc[order][[
        "product_id",
        "product_name",
        "product_class",
        "product_description"
    ]].copy()

    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores[order]

    print("Исходный запрос:", query)
    print("Использованный запрос:", used_query)
    print("После обработки:", processed)

    if changes:
        print("Исправления:", changes)

    if result.empty:
        print("Совпадений не найдено.")

    return result.reset_index(drop=True)

In [21]:
display(search_bm25("wooden coffee table"))

Исходный запрос: wooden coffee table
Использованный запрос: wooden coffee table
После обработки: wooden coffee table


,rank,product_id,product_name,product_class,product_description,score
0,1,5434,wooden coffee table,Coffee & Cocktail Tables,a rustic coffee table featuring round wooden base in quatrefoil cutout design and round wooden t...,12.082025
1,2,9364,diahann wooden coffee table,Coffee & Cocktail Tables,center your conversation space or entertainment ensemble around sleek style when you add this el...,12.056325
2,3,5371,forsman wooden coffee table,Coffee & Cocktail Tables,substantial storage and compact design of this wooden coffee table will give you a valid reason ...,12.048730
3,4,5390,treadaway wooden coffee table,Coffee & Cocktail Tables,"not only does this coffee table offer years of practical function , but it also makes an impress...",11.902603
4,5,31465,wellkamp wooden coffee table,Coffee & Cocktail Tables,"an eye-catcher set of an elegant coffee table with modish wedge-shaped ottomans , this set is en...",11.755648
5,6,5338,jayla intricately carved wooden coffee table,Coffee & Cocktail Tables,"this intricately carved wooden coffee table features a scalloped top , wooden apron with antique...",11.229840
6,7,5376,onalaska contemporary wooden coffee table,Coffee & Cocktail Tables,revamp your old furniture by replacing it with sleek and modish contemporary style furniture . b...,10.806873
7,8,5356,bellin wooden coffee table,Coffee & Cocktail Tables,"sturdy , stylish and aesthetic this craftsmanship is perfect for modern decor . this solid woode...",10.599318
8,9,5359,aquinas wooden coffee table,Coffee & Cocktail Tables,"exhibited perfectly for your transitional interiors , this coffee/cocktail table will make a cla...",10.378777
9,10,5375,choate transitional solid wooden coffee table,Coffee & Cocktail Tables,complete your decor by adorning this lovely coffee table in your living room for a stylish and t...,10.030278


In [22]:
def evaluate_bm25(
    query_ids,
    title_weight=0.7,
    fix_typos=False,
    k=10
):
    results = []

    for query_id in tqdm(query_ids, desc="Оценка BM25"):
        group = label_groups.get(query_id)

        if group is None or len(group) < 2:
            continue

        relevance = group["relevance"].to_numpy(dtype=float)

        if relevance.max() == 0:
            continue

        candidate_rows = np.array([
            product_id_to_row[product_id]
            for product_id in group["product_id"]
        ])

        scores, _, _, _ = score_query_bm25(
            query_text_by_id[query_id],
            title_weight=title_weight,
            fix_typos=fix_typos,
            candidate_rows=candidate_rows
        )

        value = ndcg_score(
            relevance.reshape(1, -1),
            scores.reshape(1, -1),
            k=k,
            ignore_ties=False
        )

        results.append({
            "query_id": query_id,
            "query": query_text_by_id[query_id],
            f"NDCG@{k}": value
        })

    return pd.DataFrame(
        results,
        columns=["query_id", "query", f"NDCG@{k}"]
    )


dev_bm25 = evaluate_bm25(
    dev_query_ids,
    fix_typos=False
)

dev_bm25_corrected = evaluate_bm25(
    dev_query_ids,
    fix_typos=True
)

Оценка BM25: 100%|██████████| 384/384 [00:01<00:00, 372.27it/s]


In [23]:
evaluations = {
    "TF-IDF": dev_plain,
    "TF-IDF + исправление": dev_corrected,
    "BM25": dev_bm25,
    "BM25 + исправление": dev_bm25_corrected
}

reference_ids = set(dev_plain["query_id"])

assert all(
    set(result["query_id"]) == reference_ids
    for result in evaluations.values()
)

comparison = pd.DataFrame([
    {
        "Вариант": name,
        "NDCG@10": result["NDCG@10"].mean(),
        "Запросов": len(result)
    }
    for name, result in evaluations.items()
])

display(
    comparison.sort_values(
        "NDCG@10", ascending=False
    ).reset_index(drop=True).round(4)
)

,Вариант,NDCG@10,Запросов
0,BM25 + исправление,0.8038,382
1,BM25,0.8031,382
2,TF-IDF + исправление,0.7817,382
3,TF-IDF,0.7815,382


BAAI/bge-small-en-v1.5

In [25]:
import gc
import time
import torch

from sentence_transformers import SentenceTransformer

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA недоступна. Выбери ядро с GPU и установленным CUDA PyTorch."
    )

DEVICE = "cuda"
BATCH_SIZE = 32

print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 5070 Ti


In [26]:
dense_product_texts = [
    (
        f"Title: {clean_text(name)}. "
        f"Category: {clean_text(category)}. "
        f"Hierarchy: {clean_text(hierarchy)}. "
        f"Description: {clean_text(description)}"
    )
    for name, category, hierarchy, description
    in products[[
        "product_name",
        "product_class",
        "category hierarchy",
        "product_description"
    ]].itertuples(index=False, name=None)
]

# Пока используем только запросы для экспериментов
dense_dev_queries = queries[
    queries["query_id"].isin(dev_query_ids)
].copy()

dense_query_texts = [
    clean_text(text)
    for text in dense_dev_queries["query"]
]

dense_query_id_to_row = {
    query_id: row
    for row, query_id in enumerate(dense_dev_queries["query_id"])
}

print("Товаров:", len(dense_product_texts))
print("Запросов:", len(dense_query_texts))

print(dense_product_texts[0][:700])

Товаров: 42994
Запросов: 384
Title: solid wood platform bed. Category: Beds. Hierarchy: Furniture / Bedroom Furniture / Beds & Headboards / Beds / Twin Beds. Description: good , deep sleep can be quite difficult to have in this busy age . fortunately , there ' s an antidote to such a problem : a nice , quality bed frame like the acacia kaylin . solidly constructed from acacia wood , this bed frame will stand the test of time and is fit to rest your shoulders on for years and years . its sleek , natural wood grain appearance provides a pleasant aesthetic to adorn any bedroom , acting both as a decorative piece as well as a place to give comfort after a hard day of work . our bed frame is designed to give ample under-bed 


In [27]:
MODEL_CONFIGS = {
    "E5-small-v2": {
        "model_id": "intfloat/e5-small-v2",
        "query_prefix": "query: ",
        "product_prefix": "passage: "
    },
    "BGE-small-en-v1.5": {
        "model_id": "BAAI/bge-small-en-v1.5",
        "query_prefix": (
            "Represent this sentence for searching relevant passages: "
        ),
        "product_prefix": ""
    }
}


def build_dense_index(model_name):
    config = MODEL_CONFIGS[model_name]

    print("Загрузка:", config["model_id"])

    model = SentenceTransformer(
        config["model_id"],
        device=DEVICE
    )

    model.max_seq_length = 512
    start = time.perf_counter()

    try:
        product_embeddings = model.encode(
            [
                config["product_prefix"] + text
                for text in dense_product_texts
            ],
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True
        )

        query_embeddings = model.encode(
            [
                config["query_prefix"] + text
                for text in dense_query_texts
            ],
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True
        )

    finally:
        model.to("cpu")
        gc.collect()
        torch.cuda.empty_cache()

    elapsed = time.perf_counter() - start

    print(f"Векторизация: {elapsed:.1f} с")

    return {
        "model": model,
        "config": config,
        "product_embeddings": product_embeddings,
        "query_embeddings": query_embeddings,
        "encoding_seconds": elapsed
    }

In [28]:
def evaluate_dense(index, k=10):
    results = []

    for query_id in tqdm(dev_query_ids, desc="Оценка"):
        group = label_groups.get(query_id)

        if group is None or len(group) < 2:
            continue

        relevance = group["relevance"].to_numpy(dtype=float)

        if relevance.max() == 0:
            continue

        candidate_rows = np.array([
            product_id_to_row[product_id]
            for product_id in group["product_id"]
        ])

        query_row = dense_query_id_to_row[query_id]
        query_vector = index["query_embeddings"][query_row]

        # Нормированные векторы:
        # скалярное произведение равно косинусному сходству
        scores = (
            index["product_embeddings"][candidate_rows]
            @ query_vector
        )

        value = ndcg_score(
            relevance.reshape(1, -1),
            scores.reshape(1, -1),
            k=k,
            ignore_ties=False
        )

        results.append({
            "query_id": query_id,
            "query": query_text_by_id[query_id],
            f"NDCG@{k}": value
        })

    return pd.DataFrame(results)

In [29]:
e5_index = build_dense_index("E5-small-v2")
dev_e5 = evaluate_dense(e5_index)

print(f"E5 NDCG@10: {dev_e5['NDCG@10'].mean():.4f}")

Загрузка: intfloat/e5-small-v2


Batches: 100%|██████████| 12/12 [00:00<00:00, 160.05it/s]


Векторизация: 23.6 с


Оценка: 100%|██████████| 384/384 [00:00<00:00, 2026.26it/s]

E5 NDCG@10: 0.8668


In [30]:
bge_index = build_dense_index("BGE-small-en-v1.5")
dev_bge = evaluate_dense(bge_index)

print(f"BGE NDCG@10: {dev_bge['NDCG@10'].mean():.4f}")

Загрузка: BAAI/bge-small-en-v1.5


Batches: 100%|██████████| 12/12 [00:00<00:00, 225.07it/s]


Векторизация: 23.2 с


Оценка: 100%|██████████| 384/384 [00:00<00:00, 1474.23it/s]

BGE NDCG@10: 0.8595


In [31]:
all_evaluations = {
    "TF-IDF": dev_plain,
    "TF-IDF + исправление": dev_corrected,
    "BM25": dev_bm25,
    "BM25 + исправление": dev_bm25_corrected,
    "E5-small-v2": dev_e5,
    "BGE-small-en-v1.5": dev_bge
}

reference_ids = set(dev_plain["query_id"])

assert all(
    set(result["query_id"]) == reference_ids
    for result in all_evaluations.values()
)

comparison = pd.DataFrame([
    {
        "Вариант": name,
        "NDCG@10": result["NDCG@10"].mean(),
        "Запросов": len(result)
    }
    for name, result in all_evaluations.items()
])

display(
    comparison.sort_values(
        "NDCG@10", ascending=False
    ).reset_index(drop=True).round(4)
)

,Вариант,NDCG@10,Запросов
0,E5-small-v2,0.8668,382
1,BGE-small-en-v1.5,0.8595,382
2,BM25 + исправление,0.8038,382
3,BM25,0.8031,382
4,TF-IDF + исправление,0.7817,382
5,TF-IDF,0.7815,382


In [32]:
def search_dense(query, index, top_k=10):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Введите непустой запрос")

    if top_k < 1:
        raise ValueError("top_k должен быть положительным")

    model = index["model"]
    prepared_query = (
        index["config"]["query_prefix"] + clean_text(query)
    )

    model.to(DEVICE)

    try:
        query_vector = model.encode(
            [prepared_query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )[0]
    finally:
        model.to("cpu")
        torch.cuda.empty_cache()

    scores = index["product_embeddings"] @ query_vector

    order = np.argsort(
        -scores, kind="stable"
    )[:top_k]

    result = products.iloc[order][[
        "product_id",
        "product_name",
        "product_class",
        "product_description"
    ]].copy()

    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores[order]

    return result.reset_index(drop=True)

In [33]:
display(
    search_dense("wooden coffee table", e5_index)
)

,rank,product_id,product_name,product_class,product_description,score
0,1,5338,jayla intricately carved wooden coffee table,Coffee & Cocktail Tables,"this intricately carved wooden coffee table features a scalloped top , wooden apron with antique...",0.906706
1,2,5434,wooden coffee table,Coffee & Cocktail Tables,a rustic coffee table featuring round wooden base in quatrefoil cutout design and round wooden t...,0.900018
2,3,5371,forsman wooden coffee table,Coffee & Cocktail Tables,substantial storage and compact design of this wooden coffee table will give you a valid reason ...,0.895801
3,4,5390,treadaway wooden coffee table,Coffee & Cocktail Tables,"not only does this coffee table offer years of practical function , but it also makes an impress...",0.891483
4,5,35893,wood coffee table,Coffee & Cocktail Tables,natural wood slices sit on a solid set of forged iron legs . cluster them as a group to create a...,0.891461
5,6,37414,matton solid wood coffee table,Coffee & Cocktail Tables,this solid wood coffee table displays a warm walnut brown finish that highlights the superior cr...,0.889358
6,7,4987,arlow solid wood block coffee table,Coffee & Cocktail Tables,"space-saving and stylish , this contemporary style wooden coffee table , featuring sleek tray-li...",0.889339
7,8,5356,bellin wooden coffee table,Coffee & Cocktail Tables,"sturdy , stylish and aesthetic this craftsmanship is perfect for modern decor . this solid woode...",0.889039
8,9,37412,henkil solid wood coffee table,,"create a warm , inviting atmosphere in your living space with the coffee table . this solid wood...",0.888901
9,10,5359,aquinas wooden coffee table,Coffee & Cocktail Tables,"exhibited perfectly for your transitional interiors , this coffee/cocktail table will make a cla...",0.886992


In [34]:
display(
    search_dense("wooden coffee table", bge_index)
)

,rank,product_id,product_name,product_class,product_description,score
0,1,5434,wooden coffee table,Coffee & Cocktail Tables,a rustic coffee table featuring round wooden base in quatrefoil cutout design and round wooden t...,0.854661
1,2,5338,jayla intricately carved wooden coffee table,Coffee & Cocktail Tables,"this intricately carved wooden coffee table features a scalloped top , wooden apron with antique...",0.846272
2,3,5371,forsman wooden coffee table,Coffee & Cocktail Tables,substantial storage and compact design of this wooden coffee table will give you a valid reason ...,0.842697
3,4,5390,treadaway wooden coffee table,Coffee & Cocktail Tables,"not only does this coffee table offer years of practical function , but it also makes an impress...",0.830902
4,5,41349,iolanthe solid wood coffee table with storage,Coffee & Cocktail Tables,our colonial-style coffee table is a timeless living accessory for your home . its wood top prov...,0.829908
5,6,5355,belhaven wooden coffee table,Coffee & Cocktail Tables,"keeping it traditional but attractive , this cocktail table will influence a warm appeal to your...",0.828579
6,7,27945,frias solid wood trestle coffee table,Patio Tables|Coffee & Cocktail Tables,add a cozy cabin style to your home with the rustic unfinished wooden coffee table . this beauti...,0.826121
7,8,5376,onalaska contemporary wooden coffee table,Coffee & Cocktail Tables,revamp your old furniture by replacing it with sleek and modish contemporary style furniture . b...,0.825477
8,9,7949,solid wood drum coffee table,Coffee & Cocktail Tables,get in touch with nature with this natural brown drum style coffee table placed in your rustic-t...,0.824760
9,10,41362,dominee solid wood coffee table with storage,Coffee & Cocktail Tables,this practical wooden coffee table is specially designed for your living room . the accent table...,0.822812


E5-base-v2 и GTE-base

In [35]:
BATCH_SIZE = 16

MODEL_CONFIGS.update({
    "E5-base-v2": {
        "model_id": "intfloat/e5-base-v2",
        "query_prefix": "query: ",
        "product_prefix": "passage: "
    },
    "GTE-base": {
        "model_id": "thenlper/gte-base",
        "query_prefix": "",
        "product_prefix": ""
    }
})

gc.collect()
torch.cuda.empty_cache()

In [36]:
e5_base_index = build_dense_index("E5-base-v2")

dev_e5_base = evaluate_dense(e5_base_index)

print(
    f"E5-base-v2 NDCG@10: "
    f"{dev_e5_base['NDCG@10'].mean():.4f}"
)

Загрузка: intfloat/e5-base-v2


Batches: 100%|██████████| 24/24 [00:00<00:00, 206.90it/s]


Векторизация: 64.5 с


Оценка: 100%|██████████| 384/384 [00:00<00:00, 1710.14it/s]

E5-base-v2 NDCG@10: 0.8732


In [37]:
gte_index = build_dense_index("GTE-base")

dev_gte = evaluate_dense(gte_index)

print(
    f"GTE-base NDCG@10: "
    f"{dev_gte['NDCG@10'].mean():.4f}"
)

Загрузка: thenlper/gte-base


Batches: 100%|██████████| 24/24 [00:00<00:00, 274.41it/s]


Векторизация: 22.1 с


Оценка: 100%|██████████| 384/384 [00:00<00:00, 633.42it/s]

GTE-base NDCG@10: 0.8640


In [38]:
all_evaluations = {
    "TF-IDF": dev_plain,
    "TF-IDF + исправление": dev_corrected,
    "BM25": dev_bm25,
    "BM25 + исправление": dev_bm25_corrected,
    "E5-small-v2": dev_e5,
    "BGE-small-en-v1.5": dev_bge,
    "E5-base-v2": dev_e5_base,
    "GTE-base": dev_gte
}

reference_ids = set(dev_plain["query_id"])

assert all(
    set(result["query_id"]) == reference_ids
    for result in all_evaluations.values()
)

comparison = pd.DataFrame([
    {
        "Вариант": name,
        "NDCG@10": result["NDCG@10"].mean(),
        "Запросов": len(result)
    }
    for name, result in all_evaluations.items()
])

comparison = comparison.sort_values(
    "NDCG@10",
    ascending=False
).reset_index(drop=True)

display(comparison.round(4))

,Вариант,NDCG@10,Запросов
0,E5-base-v2,0.8732,382
1,E5-small-v2,0.8668,382
2,GTE-base,0.8640,382
3,BGE-small-en-v1.5,0.8595,382
4,BM25 + исправление,0.8038,382
5,BM25,0.8031,382
6,TF-IDF + исправление,0.7817,382
7,TF-IDF,0.7815,382


In [39]:
test_queries = queries[
    queries["query_id"].isin(test_query_ids)
].copy()

test_query_id_to_row = {
    query_id: row
    for row, query_id in enumerate(test_queries["query_id"])
}

model = e5_base_index["model"]
config = e5_base_index["config"]

test_texts = [
    config["query_prefix"] + clean_text(query)
    for query in test_queries["query"]
]

model.to(DEVICE)

try:
    test_query_embeddings = model.encode(
        test_texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )
finally:
    model.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()

print("Тестовых запросов:", len(test_queries))

Batches: 100%|██████████| 6/6 [00:00<00:00, 198.93it/s]


Тестовых запросов: 96


In [40]:
test_results = []
skipped_queries = []

for query_id in tqdm(test_query_ids, desc="Оценка на тесте"):
    group = label_groups.get(query_id)

    if group is None or len(group) < 2:
        skipped_queries.append(query_id)
        continue

    relevance = group["relevance"].to_numpy(dtype=float)

    if relevance.max() == 0:
        skipped_queries.append(query_id)
        continue

    candidate_rows = np.array([
        product_id_to_row[product_id]
        for product_id in group["product_id"]
    ])

    query_row = test_query_id_to_row[query_id]
    query_vector = test_query_embeddings[query_row]

    scores = (
        e5_base_index["product_embeddings"][candidate_rows]
        @ query_vector
    )

    value = ndcg_score(
        relevance.reshape(1, -1),
        scores.reshape(1, -1),
        k=10,
        ignore_ties=False
    )

    test_results.append({
        "query_id": query_id,
        "query": query_text_by_id[query_id],
        "NDCG@10": value
    })

test_e5_base = pd.DataFrame(
    test_results,
    columns=["query_id", "query", "NDCG@10"]
)

print("Оценено запросов:", len(test_e5_base))
print("Пропущено запросов:", len(skipped_queries))
print(f"Тестовая NDCG@10: {test_e5_base['NDCG@10'].mean():.4f}")

Оценка на тесте: 100%|██████████| 96/96 [00:00<00:00, 1639.03it/s]

Оценено запросов: 96
Пропущено запросов: 0
Тестовая NDCG@10: 0.8439


In [41]:
final_results = pd.DataFrame([
    {
        "Модель": "E5-base-v2",
        "Выборка": "Для экспериментов",
        "NDCG@10": dev_e5_base["NDCG@10"].mean(),
        "Запросов": len(dev_e5_base)
    },
    {
        "Модель": "E5-base-v2",
        "Выборка": "Тестовая",
        "NDCG@10": test_e5_base["NDCG@10"].mean(),
        "Запросов": len(test_e5_base)
    }
])

display(final_results.round(4))

,Модель,Выборка,NDCG@10,Запросов
0,E5-base-v2,Для экспериментов,0.8732,382
1,E5-base-v2,Тестовая,0.8439,96
